# Kata 23 — Selección de Built-in Tools

> Spec: [`specs/023-builtin-tools/spec.md`](../../specs/023-builtin-tools/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

Los built-ins (Read/Write/Edit/Bash/Grep/Glob) son del **Claude Code/Agent SDK**, no del API directo. En este notebook simulo sus signatures y le doy al modelo sólo descripciones — el ejercicio es que escoja correctamente.

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cada built-in tiene un uso primario y un failure mode. Saber cuándo usar
Grep vs Read vs Edit es mecánica básica. La estrategia es incremental:
Grep → Read → Edit/Write.

## 2. Por qué importa

Read sobre todo el repo carga miles de tokens innecesarios. Edit con
anchor no único falla. Bash para algo que Grep hace nativamente añade
fricción. El examen evalúa estas decisiones rápidas.

## 3. Mock SDK — built-ins con signatures realistas

In [ ]:
import json

BUILTIN_TOOLS = [
    {"name":"Grep","description":"Searches file contents using a regex pattern within a path glob. Use to find functions, error messages, or imports.","input_schema":{"type":"object","properties":{"pattern":{"type":"string"},"glob":{"type":"string"}},"required":["pattern"]}},
    {"name":"Glob","description":"Finds files by path pattern (e.g., **/*.test.tsx). Use to enumerate files matching a name pattern, NOT for searching content.","input_schema":{"type":"object","properties":{"pattern":{"type":"string"}},"required":["pattern"]}},
    {"name":"Read","description":"Loads the full content of a single file at the given path.","input_schema":{"type":"object","properties":{"path":{"type":"string"}},"required":["path"]}},
    {"name":"Edit","description":"Replaces a unique substring in a file. Fails if old_text is not unique or not found.","input_schema":{"type":"object","properties":{"path":{"type":"string"},"old_text":{"type":"string"},"new_text":{"type":"string"}},"required":["path","old_text","new_text"]}},
    {"name":"Write","description":"Overwrites a file. Use when Edit failed due to non-unique anchor or for creating new files.","input_schema":{"type":"object","properties":{"path":{"type":"string"},"content":{"type":"string"}},"required":["path","content"]}},
    {"name":"Bash","description":"Executes a shell command. Use when no other tool fits (e.g., running tests).","input_schema":{"type":"object","properties":{"command":{"type":"string"}},"required":["command"]}},
]

queries = [
    "Hallá dónde se llama a `processRefund` en el repo de Python.",
    "Listá todos los archivos *.test.tsx del proyecto.",
    "Cargá el contenido completo de src/billing/refund.py para analizarlo.",
    "Cambiá la línea `if amount > 1000:` por `if amount > MAX_REFUND:` en src/billing/refund.py.",
    "Corré `pytest tests/` y reportá si pasa.",
]
EXPECTED = ["Grep", "Glob", "Read", "Edit", "Bash"]

print("=== selección de built-in por query ===")
results = []
for q, exp in zip(queries, EXPECTED):
    resp = client.messages.create(
        system="Eres un agente de desarrollo. Escoge la herramienta apropiada.",
        messages=[{"role":"user","content": q}],
        tools=BUILTIN_TOOLS,
        tool_choice={"type":"any"},
    )
    tu = next((b for b in resp.content if b.type == "tool_use"), None)
    actual = tu.name if tu else "(none)"
    ok = actual == exp
    print(f"  {q[:55]}... → expected={exp:6} actual={actual:6} {'OK' if ok else 'MISS'}")
    results.append(ok)
print(f"\nhit rate: {sum(results)}/{len(results)}")

## 4. Anti-patrón — failure mode de Edit

In [ ]:
# Edit con anchor no único falla. Estrategia correcta: detectar y caer a Read+Write.
fake_file = '''def login(): pass
def logout(): pass
def reset(): pass
'''

def edit_simulation(path, old_text, new_text, content):
    n = content.count(old_text)
    if n == 0: return {"error": "ANCHOR_NOT_FOUND"}
    if n > 1: return {"error": "ANCHOR_NOT_UNIQUE", "occurrences": n}
    return {"ok": True, "new_content": content.replace(old_text, new_text)}

print("Edit con anchor no único:")
print(edit_simulation("auth.py", "pass", "raise NotImplementedError()", fake_file))

print("\nEdit con anchor único:")
print(edit_simulation("auth.py", "def login(): pass", "def login(): raise NotImplementedError()", fake_file))

print("\nEstrategia: si el primer Edit retorna ANCHOR_NOT_UNIQUE, el agente debe")
print("hacer Read del archivo entero, modificar localmente, y Write completo.")

## 5. Argumento de certificación

- **Mapa de uso**: Grep (contenido), Glob (paths), Read (carga), Edit
  (modificación dirigida), Write (overwrite/fallback), Bash (último
  recurso).
- **Edit failure mode**: anchor no único → fallback Read+Write.
- **Estrategia incremental**: Grep para entry points → Read para seguir
  imports → Edit puntual. Nunca leer todo el repo upfront.
- Conexión con Kata 21: la descripción de cada built-in debe ser tan
  específica que el modelo no caiga en Bash por defecto.

## 6. Auto-evaluación

**1. ¿Qué hago cuando Edit falla por anchor no único?**

Read del archivo, modificación local, Write completo. O añadir contexto
al `old_text` (más líneas alrededor) hasta que sea único.

**2. ¿Cómo investigo un repo desconocido sin leer todos los archivos?**

Glob para mapear estructura → Grep para hallar entry points (función
main, imports clave) → Read sólo de los archivos que Grep señaló como
relevantes. Es la estrategia del Kata 19 (adaptive investigation).

**3. ¿Bash es la respuesta correcta para "buscar todos los TODOs"?**

No. Grep `pattern: "TODO"` es la opción nativa, más rápida y con
output estructurado. Bash con `grep -r TODO .` funciona pero pierde la
abstracción del SDK.